In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message.content[0].text)

In [9]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# Lesson 4. Multi-Turn conversations

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print(final_answer)

# Lesson 5. System prompts

In [ ]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

messages = []
add_user_message(messages, "Solve the equation 2x + 12 = 20")

# Without system prompt
answer = chat(messages)
print("Former", answer)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print("Latter", answer)

# L6: Temperature

In [16]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

messages = []
add_user_message(messages, "How to live ones life in a couple sentences?")

# Low temperature - more predictable
answer = chat(messages, temperature=0.0)
print("with temp=0:\n", answer)

# High temperature - more creative  
answer = chat(messages, temperature=1.0)
print("with temp=1:\n", answer)

with temp=0:
 # Living Well in Brief

**Be intentional about what matters to you**, then align your daily choices with those values rather than drifting with external pressures. **Stay curious, treat people with kindness, and accept that imperfection and change are features, not bugs.**

The rest is details.
with temp=1:
 # Living Well: The Essentials

**Be intentional about what matters to you—pursue growth, nurture relationships, and contribute something meaningful—then let go of perfection and appreciate the present moment.**

Or more simply: *Show up authentically, learn from mistakes, treat people well, and find meaning in both the big moments and small joys.*

What resonates depends on your values, but the common thread is usually: clarity + action + acceptance.


# L7: Response Streaming

In [21]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

'''
# Basic Streaming Implementation
stream = client.messages.create(
    model=model,
    max_tokens=200,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

'''

# Simplified Text Streaming
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")


# Get the complete message for database storage
final_message = stream.get_final_message()
print("\n\nFinally:\n", final_message)

# FakeDB

A lightweight in-memory database that generates and stores randomly-structured JSON records for testing and prototyping applications without needing real data infrastructure.

Finally:
 ParsedMessage(id='msg_011CghNikq5HVBZqJfL5ohQN', container=None, content=[ParsedTextBlock(citations=None, text='# FakeDB\n\nA lightweight in-memory database that generates and stores randomly-structured JSON records for testing and prototyping applications without needing real data infrastructure.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=37, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None)


# L8: Structured data

## Assistant Message Prefilling + Stop Sequences

In [2]:

ticket = "I was charged twice for my subscription this month."

message = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=50,
    stop_sequences=["</category>"],
    messages=[
        {
            "role": "user",
            "content": (
                "Classify this support ticket as one of: billing, technical, account, other.\n"
                f"Ticket: {ticket}\n"
                "Put your answer inside <category></category> tags."
            ),
        },
        {"role": "assistant", "content": "<category>"},  # prefill
    ],
)

label = message.content[0].text.strip()
print(message)
print(label)  # billing

Message(id='msg_011Ch3RXyQz8Hh6hpruTShHJ', container=None, content=[TextBlock(citations=None, text='billing', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='stop_sequence', stop_sequence='</category>', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=53, output_tokens=4, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None)
billing


# L9: Prompt evaluation
# L10: A typical eval workflow
    # Prompt Scoring using a grader